Install Dependencies

In [ ]:
!pip install -q yt-dlp openai-whisper
!apt-get -qq install ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.8/183.8 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 18.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 82.3 MB/s eta 0:00:00


In [ ]:
!ffmpeg -version

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-l

Download Video

In [ ]:
import yt_dlp

url = "https://www.youtube.com/watch?v=zCYQEHBfYuU&t=110s"

ydl_opts = {
    "format": "bestvideo+bestaudio/best",
    "merge_output_format": "mp4",
    "outtmpl": "video.%(ext)s",
    "quiet": False,
}

with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    ydl.download([url])

[youtube] Extracting URL: https://www.youtube.com/watch?v=zCYQEHBfYuU&t=110s
[youtube] zCYQEHBfYuU: Downloading webpage


[youtube] zCYQEHBfYuU: Downloading android vr player API JSON
[info] zCYQEHBfYuU: Downloading 1 format(s): 137+251
[download] Destination: video.f137.mp4
[download] 100% of  238.70MiB in 00:00:13 at 17.96MiB/s  
[download] Destination: video.f251.webm
[download] 100% of   19.53MiB in 00:00:00 at 33.32MiB/s  
[Merger] Merging formats into "video.mp4"
Deleting original file video.f137.mp4 (pass -k to keep)
Deleting original file video.f251.webm (pass -k to keep)


Extract Audio

In [ ]:
# Use FFmpeg to extract audio from the downloaded video file
!ffmpeg -i /content/video.mp4 \
    -ar 16000 \
    -ac 1 \
    /content/audio.wav \
    -y

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

Load Whisper ( to convert audio to text )

In [ ]:
import whisper

model = whisper.load_model("base")

100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 187MiB/s]


In [ ]:
import torch
print(torch.cuda.is_available())

True


Generate Transcript

In [ ]:
# Transcribe the audio file using the loaded Whisper model
result = model.transcribe(
    "/content/audio.wav",
    word_timestamps=True               # create timestamps
)

In [ ]:
# Loop through the first 10 transcription segments
for seg in result["segments"][:10]:
    print(
        f"[{seg['start']:.2f} - {seg['end']:.2f}]",
        seg['text']
    )

[0.00 - 10.12]  It's time now to dive into our next session and it gives me immense delight to invite and introduce our moderator for the session.
[10.50 - 14.12]  Ladies and gentlemen, it's none other than Sonia Shanoi.
[14.28 - 22.68]  A leading voice in Indian Business Journalism, Sonia was assistant executive editor at CNBC TV 18 for over 16 years.
[22.68 - 35.68]  A finance and FinTech expert. Today she's an independent content creator and the host of top podcasts like the Money Mindset, founder F.Ops and visionaries of India.
[36.44 - 41.98]  Please join me in welcoming Sonia Shanoi, ladies and gentlemen, a big round of applause.
[43.60 - 48.94]  Thank you to each and every one of you for being here and it is such an honour to be a part of this evening.
[48.94 - 52.76]  Because this is a topic that everyone is grappling with today.
[52.96 - 59.02]  So without wasting any time, let me invite someone who needs no introduction but I'm going to do it anyway.
[59.86 - 67.74]  Nanda Ni

Save as JSON FILE

In [ ]:
 # Save the transcript data in JSON format
import json

transcript2 = []

for seg in result["segments"]:
    transcript2.append({
        "start": seg["start"],
        "end": seg["end"],
        "text": seg["text"]
    })

with open("/content/transcript2.json", "w") as f:
    json.dump(transcript2, f, indent=4)

Download JSON FILE

In [ ]:
from google.colab import files

files.download('/content/transcript2.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!pip install contractions

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 12.9 MB/s eta 0:00:00


In [ ]:
import json

file_path = "/content/transcript2.json"

with open(file_path, "r", encoding="utf-8") as f:
    transcript = json.load(f)

print("Transcript loaded successfully!")
print("Number of segments:", len(transcript))

Transcript loaded successfully!
Number of segments: 257


In [ ]:
import json
import re
import contractions

In [ ]:
FILLERS = [
    "um", "uh", "erm", "ah", "hmm",
    "you know", "like", "sort of", "kind of"
]

def clean_text(text):

    # Expand contractions
    text = contractions.fix(text)

    # Convert to lowercase
    text = text.lower()

    # Remove filler words
    for filler in FILLERS:
        pattern = r"\b" + re.escape(filler) + r"\b"
        text = re.sub(pattern, "", text)

    # Remove unwanted punctuation
    text = re.sub(r"[^\w\s?.!]", "", text)

    # Normalize repeated punctuation
    text = re.sub(r"\?+", "?", text)
    text = re.sub(r"!+", "!", text)
    text = re.sub(r"\.+", ".", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [ ]:
cleaned_segments = []

for segment in transcript:

    cleaned_segments.append({
        "start": segment["start"],
        "end": segment["end"],
        "text": clean_text(segment["text"])
    })

print("Cleaning done:", len(cleaned_segments))

Cleaning done: 257


In [ ]:
# Save cleaned transcript
with open("cleaned_transcript.json", "w", encoding="utf-8") as f:
    json.dump({"segments": cleaned_segments}, f, indent=4)

print("Transcript cleaned successfully!")

Transcript cleaned successfully!


In [ ]:
import json

# Load cleaned transcript (list format)
with open("/content/cleaned_transcript.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# If wrapped as dict, normalize
if isinstance(data, dict) and "segments" in data:
    data = data["segments"]

qa_pairs = []

for i in range(len(data) - 1):

    current = data[i]["text"].strip()
    next_seg = data[i + 1]["text"].strip()

    # Detect question
    if current.endswith("?"):

        qa_pairs.append({
            "question": current,
            "answer": next_seg,
            "q_start": data[i]["start"],
            "q_end": data[i]["end"],
            "a_start": data[i + 1]["start"],
            "a_end": data[i + 1]["end"]
        })

print("Q/A pairs extracted:", len(qa_pairs))

Q/A pairs extracted: 5


In [ ]:
with open("qa_moments.json", "w", encoding="utf-8") as f:
    json.dump(qa_pairs, f, indent=4)

print("Saved qa_moments.json")

Saved qa_moments.json


In [ ]:
!pip install moviepy

In [ ]:
import json

video_path = "/content/video.mp4"

with open("/content/qa_moments.json", "r") as f:
    qa_moments = json.load(f)

print("Moments loaded:", len(qa_moments))

Moments loaded: 5


In [ ]:
BUFFER = 5  # seconds before and after

In [ ]:
from moviepy.editor import VideoFileClip
import os

video = VideoFileClip(video_path)

output_dir = "/content/qa_clips"
os.makedirs(output_dir, exist_ok=True)

clips = []

for i, moment in enumerate(qa_moments):

    start = max(moment["q_start"] - BUFFER, 0)
    end = moment["a_end"] + BUFFER

    clip = video.subclip(start, end)

    output_path = f"{output_dir}/qa_moment_{i}.mp4"
    clip.write_videofile(output_path, codec="libx264", audio_codec="aac")

    clips.append(output_path)

print("Clips saved:", len(clips))

/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':



Moviepy - Building video /content/qa_clips/qa_moment_0.mp4.
MoviePy - Writing audio in qa_moment_0TEMP_MPY_wvf_snd.mp4


MoviePy - Done.
Moviepy - Writing video /content/qa_clips/qa_moment_0.mp4



Moviepy - Done !
Moviepy - video ready /content/qa_clips/qa_moment_0.mp4
Moviepy - Building video /content/qa_clips/qa_moment_1.mp4.
MoviePy - Writing audio in qa_moment_1TEMP_MPY_wvf_snd.mp4


MoviePy - Done.
Moviepy - Writing video /content/qa_clips/qa_moment_1.mp4



...

Moviepy - Done !
Moviepy - video ready /content/qa_clips/qa_moment_2.mp4
Moviepy - Building video /content/qa_clips/qa_moment_3.mp4.
MoviePy - Writing audio in qa_moment_3TEMP_MPY_wvf_snd.mp4


MoviePy - Done.
Moviepy - Writing video /content/qa_clips/qa_moment_3.mp4



Moviepy - Done !
Moviepy - video ready /content/qa_clips/qa_moment_3.mp4
Moviepy - Building video /content/qa_clips/qa_moment_4.mp4.
MoviePy - Writing audio in qa_moment_4TEMP_MPY_wvf_snd.mp4


MoviePy - Done.
Moviepy - Writing video /content/qa_clips/qa_moment_4.mp4



Moviepy - Done !
Moviepy - video ready /content/qa_clips/qa_moment_4.mp4
Clips saved: 5


In [ ]:
from moviepy.editor import VideoFileClip, concatenate_videoclips

clip_paths = clips  # from your previous cell

video_clips = [VideoFileClip(c) for c in clip_paths]

highlight = concatenate_videoclips(video_clips, method="compose")

output_path = "/content/qa_highlight.mp4"

highlight.write_videofile(output_path, codec="libx264", audio_codec="aac")

Moviepy - Building video /content/qa_highlight.mp4.
MoviePy - Writing audio in qa_highlightTEMP_MPY_wvf_snd.mp4


MoviePy - Done.
Moviepy - Writing video /content/qa_highlight.mp4



Moviepy - Done !
Moviepy - video ready /content/qa_highlight.mp4


In [ ]:
from IPython.display import Video

Video(output_path, embed=True)

In [ ]:
from google.colab import files

files.download("/content/qa_highlight.mp4")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>